<a href="https://colab.research.google.com/github/SherondaWilson/MSBD566/blob/main/Real_Time_and_Streaming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests

!pip install finnhub-python plotly dash websocket-client



# replace the "demo" apikey below with your own key from alphavantage.co/support/#api-key

url = 'https://alphavantage.co/query?function=TIME_SERIES_INTRADAY&symbol=IBM&interval=5min&apikey=demo' # Corrected URL

r = requests.get(url)

data = r.json()



print(data)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 73.9 MB/s eta 0:00:00
{'Meta Data': {'1. Information': 'Intraday (5min) open, high, low, close prices and volume', '2. Symbol': 'IBM', '3. Last Refreshed': '2026-03-02 19:55:00', '4. Interval': '5min', '5. Output Size': 'Compact', '6. Time Zone': 'US/Eastern'}, 'Time Series (5min)': {'2026-03-02 19:55:00': {'1. open': '238.4000', '2. high': '238.4000', '3. low': '238.3600', '4. close': '238.3600', '5. volume': '12'}, '2026-03-02 19:50:00': {'1. open': '238.4492', '2. high': '238.5000', '3. low': '238.4492', '4. close': '238.4800', '5. volume': '25'}, '2026-03-02 19:45:00': {'1. open': '238.9900', '2. high': '238.9900', '3. low': '238.3500', '4. close': '238.3500', '5. volume': '48'}, '2026-03-02 19:40:00': {'1. open': '238.5500', '2. high': '238.8800', '3. low': '238.5000', '4. close': '238.8800', '5. volume': '5'}, '2026-03-02 19:35:00': {'1. open': '238.8500', '2. high': '238.9727', '3. low': '238.4069', '4. close': '238.4069', '

In [ ]:
!pip install requests plotly pandas --quiet

In [ ]:
import requests
import pandas as pd
import plotly.graph_objects as go
from IPython.display import clear_output
import time
from datetime import datetime

In [ ]:
# Finnhub API Key
API_KEY = "keyd6jnbkhr01qkvh5qk9tgd6jnbkhr01qkvh5qk9u0"

In [ ]:
import requests
import time
import pytz
from datetime import datetime, time as dtime
import pandas as pd

# Your Finnhub API Key
API_KEY = "d6jnbkhr01qkvh5qk9tgd6jnbkhr01qkvh5qk9u0"

#  Stocks to monitor
symbols = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA"]

eastern = pytz.timezone("US/Eastern")

def market_status():
    url = f"https://finnhub.io/api/v1/stock/market-status?exchange=US&token={API_KEY}"
    return requests.get(url).json()

def get_quote(symbol):
    url = f"https://finnhub.io/api/v1/quote?symbol={symbol}&token={API_KEY}"
    return requests.get(url).json()

def time_until_close():
    now = datetime.now(eastern)
    close_time = eastern.localize(datetime.combine(now.date(), dtime(16, 0)))
    remaining = close_time - now
    return remaining.total_seconds()

print("Monitoring Market Until Close...\n")

while True:
    status = market_status()

    if not status.get("isOpen", False):
        print("Market is currently CLOSED.")
        break

    seconds_left = time_until_close()

    if seconds_left <= 0:
        print("Market Just Closed!")
        break

    print(f"\n Time until close: {int(seconds_left//60)} minutes {int(seconds_left%60)} seconds")

    stock_data = []

    for symbol in symbols:
        quote = get_quote(symbol)
        stock_data.append({
            "Symbol": symbol,
            "Current Price": quote.get("c", 0),
            "Open": quote.get("o", 0),
            "High": quote.get("h", 0),
            "Low": quote.get("l", 0),
            "% Change from Open": round(
                ((quote.get("c",0) - quote.get("o",1)) / quote.get("o",1)) * 100, 2
            )
        })

    df = pd.DataFrame(stock_data)
    print("\n Live Market Snapshot:")
    print(df)

    time.sleep(5)

Monitoring Market Until Close...

Market is currently CLOSED.


In [ ]:
# Stocks to Track
symbols = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA"]

def get_stock_data(symbol):
    url = f"https://finnhub.io/api/v1/quote?symbol={symbol}&token={API_KEY}"
    response = requests.get(url)
    data = response.json()

    return {
        "Symbol": symbol,
        "Current Price": data.get("c", 0),
        "Open Price": data.get("o", 0),
        "High": data.get("h", 0),
        "Low": data.get("l", 0),
        "Previous Close": data.get("pc", 0),
        "Timestamp": datetime.now()
    }

def calculate_percentage_change(current, open_price):
    if open_price == 0:
        return 0
    return ((current - open_price) / open_price) * 100

print("Starting Real-Time Stock Dashboard...\n")

while True:
    stock_list = []

    for symbol in symbols:
        stock_data = get_stock_data(symbol)
        stock_data["% Change"] = calculate_percentage_change(
            stock_data["Current Price"],
            stock_data["Open Price"]
        )
        stock_list.append(stock_data)

    df = pd.DataFrame(stock_list)

    # Color Coding
    colors = ['green' if x >= 0 else 'red' for x in df["% Change"]]

    # Clear previous output
    clear_output(wait=True)

    fig = go.Figure(data=[
        go.Bar(
            x=df["Symbol"],
            y=df["% Change"],
            marker_color=colors,
            text=[f"{x:.2f}%" for x in df["% Change"]],
            textposition='outside'
        )
    ])

    fig.update_layout(
        title="Real-Time Stock % Change from Opening Price",
        yaxis_title="% Change",
        xaxis_title="Stock Symbol",
        template="plotly_white",
        yaxis=dict(zeroline=True, zerolinewidth=2, zerolinecolor='black')
    )

    fig.show()

    time.sleep(5)

In [ ]:
import plotly.graph_objects as go

from plotly.subplots import make_subplots



# Color coding: Green up, Red down

colors = ['green' if x >= 0 else 'red' for x in df_quotes['pct_change']]



fig = go.Figure(data=[

  go.Bar(

    x=df_quotes['symbol'],

    y=df_quotes['current'],

    marker_color=colors,

    text=[f"${v:.2f}<br>{p:+.1f}%" for v,p in zip(df_quotes['current'], df_quotes['pct_change'])],

    textposition='auto',

    hovertemplate='<b>%{x}</b><br>Price: $%{y:.2f}<br>Change: %{text}<extra></extra>'

  )

])



fig.update_layout(

  title="Live Stock Prices & % Change from Open",

  xaxis_title="Symbol", yaxis_title="Price ($)",

  template="plotly_dark",

  height=500

)

fig.show()



In [ ]:
def get_candlestick(symbol):
    url = f"https://finnhub.io/api/v1/quote?symbol={symbol}&token={API_KEY}"
    response = requests.get(url).json()

    fig = go.Figure(data=[go.Candlestick(
        x=[datetime.now()],
        open=[response['o']],
        high=[response['h']],
        low=[response['l']],
        close=[response['c']]
    )])

    fig.update_layout(title=f"{symbol} Live Candlestick")
    fig.show()

get_candlestick("AAPL")

In [ ]:
for index, row in df.iterrows():
    if abs(row["% Change"]) > 2:
        print(f"ALERT  {row['Symbol']} moved {row['% Change']:.2f}%")

A similar architecture can be applied to continuous patient monitoring systems in hospitals or ICU settings.

In [ ]:
import plotly.graph_objects as go
from IPython.display import clear_output, display
import time

def draw_bio_chart(buffer_df):
    fig = go.Figure()

    for signal in buffer_df["signal"].unique():
        subset = buffer_df[buffer_df["signal"] == signal]

        fig.add_trace(
            go.Scatter(
                x=subset["time"],
                y=subset["value"],
                mode="lines",
                name=signal
            )
        )

    # Clinical threshold lines
    if not buffer_df.empty:
        xmin = buffer_df["time"].min()
        xmax = buffer_df["time"].max()

        # Heart rate warning threshold
        fig.add_shape(
            type="line",
            x0=xmin, x1=xmax,
            y0=120, y1=120,
            xref="x", yref="y",
            line=dict(dash="dash")
        )

    fig.update_layout(
        title="Real-Time Biomedical Monitoring Dashboard (Last 5 Minutes)",
        xaxis_title="Time",
        yaxis_title="Measured Value",
        template="plotly_white",
        height=600
    )

    clear_output(wait=True)
    display(fig)